# [Step 5 - CSVLoader] One row, one Document

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

### What you'll learn

- how `CSVLoader` turns each CSV row into a `Document` of `"column: value"` lines
- controlling fields with `content_columns` vs `metadata_columns`
- what `source_column` really does to `metadata["source"]` (and when to avoid it)
- filtering loaded Documents by metadata - the skill the capstone retrieval will reuse

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================

# --- Standard library -------------------------------------------------
import os                      # file-system odds and ends
import urllib.request          # polite HTTP fetching
from pathlib import Path       # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import CSVLoader

# ---------------------------------------------------------------------
# TRACK WALKER - resolve 03_agentic_ai by walking upward from cwd.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - fetch once, cache under DATA, reuse forever.
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    target.write_bytes(payload)
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; drops a BOM if present."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe printing for arbitrary text."""
    return text.encode("ascii", errors="replace").decode("ascii")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_38184\1861760523.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


### 1. Why CSV needs its own loader

A spreadsheet is not prose - it is many small, self-contained facts.
The natural retrieval unit is the ROW ("which wine has alcohol 9.4 and
fixed acidity 7.3?"), so `CSVLoader` emits **one Document per data row**:

- `page_content` = every column rendered as readable `"column: value"` lines
  (this text is what gets embedded later);
- `metadata` = `{"source": <file path>, "row": <row index>}` plus any columns
  you promote via `metadata_columns`.

Our specimen: the UCI red-wine-quality dataset (1599 physicochemical rows,
semicolon-separated). Note the delimiter - European-style exports love `;`,
and Python's csv reader defaults to `,`, so we must SAY so explicitly.

In [2]:
WINE_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)
wine_path = DATA / "winequality-red.csv"
get_bytes("winequality-red.csv", WINE_URL)     # download-once into the cache

# Baseline load: EVERY column becomes content; delimiter declared via csv_args,
# which passes straight through to Python's csv.DictReader.
basic_loader = CSVLoader(str(wine_path), csv_args={"delimiter": ";"})
basic_docs = basic_loader.load()

print(f"Documents (rows) loaded : {len(basic_docs)}")
print(f"first row metadata      : {basic_docs[0].metadata}")
print("--- first row page_content ---")
print(to_ascii(basic_docs[0].page_content))

[cache] winequality-red.csv: 84,199 bytes
Documents (rows) loaded : 1599
first row metadata      : {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\winequality-red.csv', 'row': 0}
--- first row page_content ---
fixed acidity: 7.4
volatile acidity: 0.7
citric acid: 0
residual sugar: 1.9
chlorides: 0.076
free sulfur dioxide: 11
total sulfur dioxide: 34
density: 0.9978
pH: 3.51
sulphates: 0.56
alcohol: 9.4
quality: 5


### 2. Curating fields: content_columns vs metadata_columns

Embedding all eleven chemical columns makes every row's vector similar to
every other's - numbers dominate the meaning. For retrieval it is smarter
to keep FEWER columns as searchable content and move the rest into metadata:

- `content_columns=[...]` - ONLY these columns form `page_content`;
- `metadata_columns=[...]` - these ride along as filterable metadata keys
  instead of polluting the embedded text.

Below: embed acidity + alcohol (the interesting chemistry), keep `quality`
as a metadata facet.

In [3]:
showcase_loader = CSVLoader(
    str(wine_path),
    csv_args={"delimiter": ";"},
    content_columns=["fixed acidity", "alcohol"],   # becomes page_content
    metadata_columns=["quality"],                   # becomes metadata["quality"]
)
showcase_docs = showcase_loader.load()

d = showcase_docs[0]
print(f"still one Document per row : {len(showcase_docs)}")
print(f"page_content now minimal   : {to_ascii(d.page_content)!r}")
print(f"metadata gained 'quality'  : {d.metadata}")

still one Document per row : 1599
page_content now minimal   : 'fixed acidity: 7.4\nalcohol: 9.4'
metadata gained 'quality'  : {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\winequality-red.csv', 'row': 0, 'quality': '5'}


### 3. The `source_column` twist

By default `metadata["source"]` holds the FILE PATH. Pass `source_column=...`
and it gets REPLACED by that row's cell value. That is perfect when a column
genuinely identifies origin (a URL or filename column). Here we demo it with
`quality` purely to see the mechanics - and flag why it is usually a bad idea
to repurpose `source` this way: provenance tools expect a location there.

In [4]:
source_demo_loader = CSVLoader(
    str(wine_path),
    csv_args={"delimiter": ";"},
    source_column="quality",       # metadata['source'] <- row's quality cell
)
source_demo_docs = source_demo_loader.load()
print(f"default source was : ...winequality-red.csv")
print(f"now source reads   : {source_demo_docs[0].metadata['source']!r}"
      "   <- the ROW's quality, not a path")

default source was : ...winequality-red.csv
now source reads   : '5'   <- the ROW's quality, not a path


### 4. One meaningful manipulation: filter by metadata

This is THE pattern the capstone reuses after retrieval: keep Documents whose
metadata satisfies a predicate. High-quality wines (quality >= 7) are rare -
let us find them without touching any database.

In [5]:
high_quality = [doc for doc in showcase_docs if int(doc.metadata["quality"]) >= 7]
share = 100.0 * len(high_quality) / len(showcase_docs)
print(f"rows with quality >= 7 : {len(high_quality)} of {len(showcase_docs)} ({share:.1f}%)")
print("\n--- sample of a filtered row ---")
print(to_ascii(high_quality[0].page_content))
print(high_quality[0].metadata)

# Bonus aggregation: quality distribution straight from metadata.
from collections import Counter
distribution = Counter(doc.metadata["quality"] for doc in showcase_docs)
print("\nquality histogram:", dict(sorted(distribution.items())))

rows with quality >= 7 : 217 of 1599 (13.6%)

--- sample of a filtered row ---
fixed acidity: 7.3
alcohol: 10
{'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\winequality-red.csv', 'row': 7, 'quality': '7'}

quality histogram: {'3': 10, '4': 53, '5': 681, '6': 638, '7': 199, '8': 18}


### 5. Pitfalls worth remembering

**Pitfall - wrong delimiter**: with default `,` parsing a `;`-separated file
loads as ONE giant column per row (or one giant row overall). Always pass
`csv_args={"delimiter": ";"}` when needed - and inspect row 1 afterwards.

**Pitfall - row-vs-document is a design choice**: per-row Documents shine for
lookup questions but cannot answer "compare all 1599 wines" - that needs the
whole table (or an agent with tools). Decide per use-case, not by habit.

**Pro-tip**: quoted headers like `"fixed acidity"` are handled by Python's
csv module automatically - parsed keys come out clean (`fixed acidity`),
which is exactly why `content_columns` above matches without quotes.

### Takeaway

**CSVLoader = one Document per row, `column: value` content, free-form
metadata. Split columns between `content_columns` (embedded) and
`metadata_columns` (filterable), then treat metadata filters as your
first-class retrieval lever.**

### Summary

- 1599 rows became 1599 Documents; `csv_args={"delimiter": ";"}` was
  mandatory for this UCI export.
- `content_columns` / `metadata_columns` curate what gets embedded versus
  what stays queryable metadata.
- `source_column` swaps `metadata["source"]` to a cell value - useful for
  real identifiers, misleading elsewhere.
- Metadata filtering (`quality >= 7`) kept 217 of 1599 rows - the same
  predicate pattern powers filtered retrieval in the capstone.